In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import xarray as xr

from fast_o3_iv import utils, metrics
import fast_o3_iv.bias_correction as bc
from fast_o3_iv.config import INTERIM_DATA_DIR
from fast_o3_iv.modeling.train import DatasetConfig


In [ ]:
# load observation osdma8
ramp_osdma8 = xr.open_dataset((
        INTERIM_DATA_DIR
        / "RAMP"
        / "Jacob_wRAMP_OSDMA8_1990_2017.nc"))


In [ ]:
# load modeled monthly mda8o3
config_hist = DatasetConfig(
            source_id = 'CESM2',
            cmip_experiment_id = 'historical',
            variable_setname = 'setC',
            year_range = range(1975,2015)
        )

hist_anm = utils.load_or_aggregate_results(
    "inference",
    config_hist,
     "GPR-local",
)

In [ ]:
config_ssp = DatasetConfig(
            source_id = 'CESM2',
            cmip_experiment_id = 'ssp585',
            variable_setname = 'setC',
            year_range = range(2015,2055)
        )
ssp_anm = utils.load_or_aggregate_results(
    "inference",
    config_ssp,
     "GPR-local",
)

In [ ]:
# check that seasonal cycle is removed from original anomalies
hist_anm['y_pred'].isel(ensemble_member=0, lat=0, lon=0).groupby('time.month').mean().plot()

In [ ]:
# add climatology (only required if target == mda8o3_anm (setC))

train_config_hist = DatasetConfig(
            source_id = 'CESM2-WACCM6',
            cmip_experiment_id = 'historical',
            variable_setname = 'setC',
            year_range = range(1975,2015)
        )
train_config_ssp585 = DatasetConfig(
            source_id = 'CESM2-WACCM6',
            cmip_experiment_id = 'ssp585',
            variable_setname = 'setC',
            year_range = range(2015,2055)
        )

hist = utils.add_mda8o3_climatology(hist_anm, train_config_hist) 
ssp585 = utils.add_mda8o3_climatology(ssp_anm, train_config_ssp585) 

In [ ]:
# check that seasonal cycle is fixed
hist['y_pred'].isel(ensemble_member=0, lat=0, lon=0).groupby('time.month').mean().plot()

In [ ]:
#convert monthly mda8o3 to osdma8 
hist_osdma8 = metrics.calc_osdma8(hist)
ssp585_osdma8 = metrics.calc_osdma8(ssp585)

In [ ]:
# calculate scaling factor based on historical mod vs obs osdma8
scaling_factor, obs_base, mod_base = bc.get_bc_scaling_factor(hist_osdma8, ramp_osdma8)

In [ ]:
# apply bias correction
hist_osdma8_bc = bc.bias_correct(hist_osdma8, scaling_factor)
ssp585_osdma8_bc = bc.bias_correct(ssp585_osdma8, scaling_factor)

In [ ]:
# save to netcdf
#file_path = bc.return_osdma8_bc_path("inference", config_hist, "GPR-local")
#hist_osdma8_bc.to_netcdf(file_path)

file_path = bc.return_osdma8_bc_path("inference", config_ssp, "GPR-local")
ssp585_osdma8_bc.to_netcdf(file_path)